# The Classification Notebook

#### This process requires building a `a golden dataset` which can be used to use the LLM classification methodology. To calculate the number of entries to include in the golden dataset, I used Naël Shiab's calculator publised on observablehq, you can find it here: 

https://observablehq.com/@nshiab/how-many-entries-should-i-double-check-when-using-ai 

#### The calculator told me: for a 95% confidence rate, with a 5% margin of error I needed to double-check 336 random results. I hand-coded a schema for the 336 entries, classifying them in themes and extracting keywords like 'Iranian economy', 'protests', 'foreign intervention', 'regional politics', 'Gaza', etc. 

In [ ]:
!pip install gspread google-auth


In [ ]:
!pip install scikit-learn

In [ ]:
!pip install openai diskcache 

In [27]:
#importing what i might need

import pandas as pd
import csv, requests, os
import numpy as np
import sys 
import tqdm
from dotenv import load_dotenv
from sklearn import *
import openai
import diskcache

In [ ]:
#taking out the sample of 336 from the CSV
df = pd.read_csv('explosive_media_messages.csv', encoding='utf-8-sig')
golden = df.sample(n=336, random_state=42, ignore_index=False)
golden.to_csv('golden_dataset.csv', index=True, index_label='original_row')

print(f"Sampled {len(golden)} rows")
golden.head()

#i put this in a new sheet in my google sheet and called golden_dataset 


Sampled 336 rows


,message_text_persian,message_text_english,time_est,date,has_media,media_filename,audio_transcription_persian,audio_transcription_persian_v2,audio_transcription_english,ocr_text_persian,ocr_text_english,keywords,screenshots,theme,include_person,AI_generated
415,بحران آب هنوز پابرجاست!🥲\n\n🔻 رئیس شورای شهر ت...,Water crisis still persists!🥲 Tehran City Coun...,2026-01-06 13:17:55 EST,2026-01-06,Y,2026-01-06_053.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2232,🛑 بازی چیشد؟\n\n⚽️ مس ۰ _ ۱ استقلال \n\n🆔 @akh...,🛑 How was the game?\n\n⚽️ Mes 0 - 1 Esteghlal\...,2026-02-22 12:40:49 EST,2026-02-22,Y,2026-02-22_056.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1149,🛑 کوتاه نمیایم!\n\n🔻 نخست‌وزیر اسپانیا گفته:\n...,🛑 We won't back down!\n\n🔻 Spain's Prime Minis...,2026-02-05 12:16:59 EST,2026-02-05,Y,2026-02-05_053.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1113,🛑 المپیک شروع نشده، برقش رفت\n\n🎿 ۵ دقیقه بعد ...,"🛑 Olympics hasn't started, power went out\n\n🎿...",2026-02-05 04:46:30 EST,2026-02-05,Y,2026-02-05_019.mp4,Bruce Martin. Big Wheat. Trying to move the st...,برسمادده. بکویت. کنه موزه استوانزه پرمدرنگز. ر...,[Audio unintelligible - heavily corrupted with...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1225,🇮🇷 ورود کاروان ورزشی ایران به مراسم افتتاحیهٔ...,🇮🇷 Iran's sports delegation entering the openi...,2026-02-06 23:50:07 EST,2026-02-06,Y,2026-02-06_062.mp4,I would love to. I would love to. She would be...,این این این این این این این این این این این ای...,This this this this this this this this this t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#making sure i can later connect it to the google sheet

def make_regular_gsheet_url(doc_id, sheet_id):
    return f"https://docs.google.com/spreadsheets/d/{doc_id}/edit#gid={sheet_id}"

def make_csv_gsheet_url(doc_id, sheet_id):
    return f"https://docs.google.com/spreadsheets/d/{doc_id}/export?format=csv&id={doc_id}&gid={sheet_id}"


In [28]:
#connecting my google sheet to this notebook 

GOOGLE_SHEET_ID = '1Gz7KgWNzbAcUG5R8LmGZ7lRn41orQKnbEzTc-mt6Zb0'  
SHEET_GID = '783599104'                       


google_sheet_url = make_regular_gsheet_url(GOOGLE_SHEET_ID, SHEET_GID)
print("Querying Doc:", google_sheet_url)

google_sheet_csv_url = make_csv_gsheet_url(GOOGLE_SHEET_ID, SHEET_GID)
response = requests.get(google_sheet_csv_url)
response.raise_for_status()

reader = csv.reader(response.text.splitlines())
header = next(reader)
df = pd.DataFrame(list(reader), columns=header)

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(df.columns.tolist())
df.head()

Querying Doc: https://docs.google.com/spreadsheets/d/1Gz7KgWNzbAcUG5R8LmGZ7lRn41orQKnbEzTc-mt6Zb0/edit#gid=783599104
Loaded 6097 rows, 17 columns
['original_row', 'message_text_persian', 'message_text_english', 'time_est', 'date', 'has_media', 'media_filename', 'audio_transcription_persian', 'audio_transcription_persian_v2', 'audio_transcription_english', 'ocr_text_persian', 'ocr_text_english', 'keywords', 'screenshots', 'theme', 'include_person', 'AI_generated']


,original_row,message_text_persian,message_text_english,time_est,date,has_media,media_filename,audio_transcription_persian,audio_transcription_persian_v2,audio_transcription_english,ocr_text_persian,ocr_text_english,keywords,screenshots,theme,include_person,AI_generated
0,415,Ø¨Ø­Ø±Ø§Ù Ø¢Ø¨ ÙÙÙØ² Ù¾Ø§Ø¨Ø±Ø¬Ø§Ø³Øª!ð¥²...,Water crisis still persists!ð¥² Tehran City C...,2026-01-06 13:17:55 EST,2026-01-06,Y,2026-01-06_053.jpg,,,,,,,,,,
1,2232,ð Ø¨Ø§Ø²Û ÚÛØ´Ø¯Øâ½ï¸ ÙØ³ Û° _ Û± Ø§Ø...,ð How was the game?â½ï¸ Mes 0 - 1 Esteghl...,2026-02-22 12:40:49 EST,2026-02-22,Y,2026-02-22_056.jpg,,,,,,,,,,
2,1149,ð Ú©ÙØªØ§Ù ÙÙÛØ§ÛÙ!ð» ÙØ®Ø³ØªâÙØ...,ð We won't back down!ð» Spain's Prime Min...,2026-02-05 12:16:59 EST,2026-02-05,Y,2026-02-05_053.jpg,,,,,,,,,,
3,1113,ð Ø§ÙÙÙ¾ÛÚ© Ø´Ø±ÙØ¹ ÙØ´Ø¯ÙØ Ø¨Ø±ÙØ´ ...,"ð Olympics hasn't started, power went outð...",2026-02-05 04:46:30 EST,2026-02-05,Y,2026-02-05_019.mp4,Bruce Martin. Big Wheat. Trying to move the st...,Ø¨Ø±Ø³Ù,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ø§Ø¯Ø¯Ù. Ø¨Ú©ÙÛØª. Ú©ÙÙ Ù,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
#adding the llm-classifier

%cd /Users/towcenter/Desktop/explosive-media-footage
!~/.pyenv/versions/3.11.14/bin/python classify_llm.py

/Users/towcenter/Desktop/explosive-media-footage
Using 25 few-shot examples across 17 themes
2504 rows to classify
  [10/2504] batch 1/251
  [20/2504] batch 2/251
  [30/2504] batch 3/251
  [40/2504] batch 4/251
  [50/2504] batch 5/251
  [60/2504] batch 6/251
  [70/2504] batch 7/251
  [80/2504] batch 8/251
  [90/2504] batch 9/251
  [100/2504] batch 10/251
  [110/2504] batch 11/251
  [120/2504] batch 12/251
  [130/2504] batch 13/251
  [140/2504] batch 14/251
  [150/2504] batch 15/251
  [160/2504] batch 16/251
  [170/2504] batch 17/251
  [180/2504] batch 18/251
  [190/2504] batch 19/251
  [200/2504] batch 20/251
  [210/2504] batch 21/251
  [220/2504] batch 22/251
  [230/2504] batch 23/251
  [240/2504] batch 24/251
  [250/2504] batch 25/251
  [260/2504] batch 26/251
  [270/2504] batch 27/251
  [280/2504] batch 28/251
  [290/2504] batch 29/251
  [300/2504] batch 30/251
  [310/2504] batch 31/251
  [320/2504] batch 32/251
  [330/2504] batch 33/251
  [340/2504] batch 34/251
  [350/2504] batch 

### What was the anti regime content?

In [2]:
  
CSV = "/Users/towcenter/Desktop/explosive-media-footage/explosive-media-analysis/CSVs/explosive_media_messages.csv"                                                                                               
                                                                                                                                                                                                         
df = pd.read_csv(CSV, encoding="utf-8-sig")                                                                                                                                                                       
                                                                                                                                                                                                           
# LLM classification lives in `theme`; `keywords` sometimes carries the tag too.                                                                                                                                  
theme = df["theme"].fillna("").str.strip()                                                                                                                                                               
kw    = df["keywords"].fillna("").str.lower()                                                                                                                                                                     
                                                                                                                                                                                                          
anti_regime_mask = (theme == "Anti-regime") | kw.str.contains(r"anti[- ]regime", regex=True)                                                                                                                      
                                                                                                                                                                                                           
# Everything tagged anti-regime                                                                                                                                                                                   
anti = df[anti_regime_mask].copy()                                                                                                                                                                       
                                                                                                                                                                                                                    
# Just the rows that actually have footage attached (video/image)                                                                                                                                                 
footage = anti[anti["media_filename"].fillna("").str.lower()                                                                                                                                                      
                       .str.endswith((".mp4", ".jpg", ".jpeg", ".png"))].copy()                                                                                                                                     
                                                                                                                                                                                                                    
# Convenience column: video vs image                                                                                                                                                                              
footage["media_type"] = footage["media_filename"].str.lower().apply(                                                                                                                                              
  lambda f: "video" if f.endswith(".mp4") else "image"                                                                                                                                                          
  )                                                                                                                                                                                                        
                                                                                                                                                                                                                    
# Sort chronologically and keep the useful columns                                                                                                                                                                
footage = footage.sort_values("date")[[                                                                                                                                                                 
      "date", "time_est", "media_type", "media_filename",                                                                                                                                                           
      "theme", "keywords", "message_text_english",                                                                                                                                                        
  ]]                                                                                                                                                                                                                
                                                                                                                                                                                                                    
print(f"Anti-regime rows total: {len(anti)}")                                                                                                                                                           
print(f"  ...with footage:      {len(footage)}")                                                                                                                                                                  
print(f"  videos: {(footage.media_type == 'video').sum()}, "                                                                                                                                                      
        f"images: {(footage.media_type == 'image').sum()}")                                                                                                                                                         
                                                                                                                                                                                                                    
# Save to CSV for downstream use                                                                                                                                                                                  
out = "/Users/towcenter/Desktop/explosive-media-footage/anti_regime_footage.csv"                                                                                                                        
footage.to_csv(out, index=False)                                                                                                                                                                                  
print(f"Wrote {out}")

Anti-regime rows total: 85
  ...with footage:      80
  videos: 47, images: 33
Wrote /Users/towcenter/Desktop/explosive-media-footage/anti_regime_footage.csv


### What was the content about artificial intelligence? 

In [3]:
import pandas as pd                                                                                                                                                                                      
                                                                                                                                                                                                                    
CSV = "/Users/towcenter/Desktop/explosive-media-footage/explosive-media-analysis/CSVs/explosive_media_messages.csv"                                                                                               
                                                                                                                                                                                                                    
df = pd.read_csv(CSV, encoding="utf-8-sig")                                                                                                                                                                       
                                                                                                                                                                                                          
# LLM classification in `theme`; keywords carry related AI tags too (same as build.py)                                                                                                                            
theme = df["theme"].fillna("").str.strip()                                                                                                                                                               
kw    = df["keywords"].fillna("").str.lower()                                                                                                                                                                     
                                                                                                                                                                                                           
ai_kw_pattern = r"ai[- ]generated|using ai|commentary on ai|detecting ai|gemini"                                                                                                                                  
                                                                                                                                                                                                                    
using_ai_mask = (theme == "Using AI") | kw.str.contains(ai_kw_pattern, regex=True)                                                                                                                                
                                                                                                                                                                                                          
# Everything tagged Using AI                                                                                                                                                                                      
ai_posts = df[using_ai_mask].copy()                                                                                                                                                                      
                                                                                                                                                                                                                    
# Rows with footage attached (video/image)                                                                                                                                                                        
media = ai_posts["media_filename"].fillna("").str.lower()                                                                                                                                               
footage = ai_posts[                                                                                                                                                                                               
    media.str.endswith(".mp4") |                                                                                                                                                                        
    media.str.endswith(".jpg") |                                                                                                                                                                                  
    media.str.endswith(".jpeg") |                                                                                                                                                                        
    media.str.endswith(".png")                                                                                                                                                                                    
].copy()                                                                                                                                                                                                
                                                                                                                                                                                                          
# Convenience column                                                                                                                                                                                     
footage["media_type"] = footage["media_filename"].str.lower().apply(                                                                                                                                    
    lambda f: "video" if f.endswith(".mp4") else "image"                                                                                                                                                          
)                                                                                                                                                                                                       
                                                                                                                                                                                                                    
# Flag which ones the LLM also marked as AI-generated content                                                                                                                                                     
footage["is_ai_generated"] = footage["AI_generated"].fillna("").str.strip() == "Yes"                                                                                                                   
                                                                                                                                                                                                                    
# Sort chronologically, keep useful columns                                                                                                                                                                       
footage = footage.sort_values("date")[[                                                                                                                                                                 
    "date", "time_est", "media_type", "media_filename",                                                                                                                                                           
    "theme", "keywords", "is_ai_generated", "message_text_english",                                                                                                                                               
]]                                                                                                                                                                                                      
                                                                                                                                                                                                                    
print(f"'Using AI' rows total:  {len(ai_posts)}")                                                                                                                                                       
print(f"  ...with footage:      {len(footage)}")                                                                                                                                                                  
print(f"  videos: {(footage.media_type == 'video').sum()}, "                                                                                                                                                      
    f"images: {(footage.media_type == 'image').sum()}")                                                                                                                                                         
print(f"  AI-generated content: {footage.is_ai_generated.sum()}")                                                                                                                                                 
                                                                                                                                                                                                                    
out = "/Users/towcenter/Desktop/explosive-media-footage/using_ai_footage.csv"                                                                                                                           
footage.to_csv(out, index=False)                                                                                                                                                                                  
print(f"Wrote {out}")

'Using AI' rows total:  87
  ...with footage:      85
  videos: 20, images: 65
  AI-generated content: 8
Wrote /Users/towcenter/Desktop/explosive-media-footage/using_ai_footage.csv
